**versione llama_cloud usata**: 2026-09-13

note: se riesegui il codice con "latest", verificare che la versione di llama_cloud sia corretta

In [1]:
import os
import re
import sys
from pathlib import Path

import llama_cloud
from llama_cloud import LlamaCloud
from tqdm.auto import tqdm

c:\venvs\gbv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FOLDER = Path("../documents")
OUTPUT_FOLDER = Path("./1_parsed_results")

PARSER_TIER = "agentic"
PARSER_VERSION = "2026-09-13" # latest

# False = se il Markdown esiste già, salta il documento
# True  = rigenera anche i documenti già processati
OVERWRITE = False

In [3]:
# ============================================================
# ENVIRONMENT INFO
# ============================================================

print("Python:", sys.executable)
print("llama_cloud:", llama_cloud.__file__)
print(
    "llama_cloud version:",
    getattr(llama_cloud, "__version__", "versione non disponibile"),
)

Python: c:\venvs\gbv\Scripts\python.exe
llama_cloud: c:\venvs\gbv\Lib\site-packages\llama_cloud\__init__.py
llama_cloud version: 2.16.0


In [4]:
# ============================================================
# LLAMA CLOUD CLIENT
# ============================================================
api_key = os.environ.get("LLAMA_CLOUD_API_KEY_MOLIVERI") #LLAMA_CLOUD_API_KEY

client = LlamaCloud(
    api_key=api_key,
    base_url="https://api.cloud.eu.llamaindex.ai",
)

In [5]:
# ============================================================
# HELPERS
# ============================================================

def natural_sort_key(path: Path):
    """
    Natural sorting:
    en_doc_1.pdf
    en_doc_2.pdf
    en_doc_10.pdf
    """
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", path.name)
    ]


def output_path_for(pdf_path: Path) -> Path:
    """
    Convert:
        ../documents/en_doc_1.pdf

    into:
        ./1_parsed_results/en_doc_1.md
    """
    return OUTPUT_FOLDER / f"{pdf_path.stem}.md"


def upload_and_parse(pdf_path: Path):
    """
    Upload a PDF to LlamaCloud and parse it as Markdown.
    """

    file_obj = client.files.create(
        file=str(pdf_path),
        purpose="parse",
    )

    result = client.parsing.parse(
        file_id=file_obj.id,
        tier=PARSER_TIER,
        version=PARSER_VERSION,
        expand=["markdown_full"],
    )

    return result


def save_result(result, output_path: Path):
    """
    Save Markdown returned by LlamaParse.
    """

    markdown = result.markdown_full or ""

    output_path.write_text(
        markdown,
        encoding="utf-8",
    )

    return len(markdown)

In [6]:
# ============================================================
# FIND PDFs
# ============================================================

if not INPUT_FOLDER.exists():
    raise FileNotFoundError(
        f"Input folder not found: {INPUT_FOLDER.resolve()}"
    )

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)

pdf_files = sorted(
    INPUT_FOLDER.glob("*.pdf"),
    key=natural_sort_key,
)

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {INPUT_FOLDER.resolve()}"
    )


print(f"\nFound {len(pdf_files)} PDFs:\n")

for pdf_path in pdf_files:
    print(pdf_path.name)


Found 17 PDFs:

en_doc_1.pdf
en_doc_2.pdf
en_doc_3.pdf
en_doc_4.pdf
en_doc_5.pdf
en_doc_6.pdf
en_doc_7.pdf
en_doc_8.pdf
en_doc_9.pdf
en_doc_10.pdf
en_doc_11.pdf
en_doc_12.pdf
en_doc_23.pdf
en_doc_24.pdf
en_doc_25.pdf
en_doc_26.pdf
en_doc_27.pdf


In [7]:
# ============================================================
# PARSE
# ============================================================

processed = 0
skipped = 0
failed = 0


for pdf_path in tqdm(pdf_files, desc="Parsing PDFs"):

    output_path = output_path_for(pdf_path)

    # --------------------------------------------------------
    # Resume support
    # --------------------------------------------------------

    if output_path.exists() and not OVERWRITE:
        tqdm.write(
            f"SKIP  {pdf_path.name} "
            f"-> {output_path.name} already exists"
        )

        skipped += 1
        continue

    # --------------------------------------------------------
    # Parse
    # --------------------------------------------------------

    try:
        result = upload_and_parse(pdf_path)

        chars = save_result(
            result,
            output_path,
        )

        tqdm.write(
            f"OK    {pdf_path.name} "
            f"-> {output_path.name} "
            f"({chars:,} chars)"
        )

        processed += 1

    except Exception as exc:

        tqdm.write(
            f"ERROR {pdf_path.name}: "
            f"{type(exc).__name__}: {exc}"
        )

        failed += 1

Parsing PDFs:   0%|          | 0/17 [00:00<?, ?it/s]

SKIP  en_doc_1.pdf -> en_doc_1.md already exists
SKIP  en_doc_2.pdf -> en_doc_2.md already exists
SKIP  en_doc_3.pdf -> en_doc_3.md already exists
SKIP  en_doc_4.pdf -> en_doc_4.md already exists
SKIP  en_doc_5.pdf -> en_doc_5.md already exists
SKIP  en_doc_6.pdf -> en_doc_6.md already exists
SKIP  en_doc_7.pdf -> en_doc_7.md already exists
SKIP  en_doc_8.pdf -> en_doc_8.md already exists
SKIP  en_doc_9.pdf -> en_doc_9.md already exists


Parsing PDFs:  59%|█████▉    | 10/17 [01:43<01:12, 10.40s/it]

OK    en_doc_10.pdf -> en_doc_10.md (1,243,520 chars)


Parsing PDFs:  65%|██████▍   | 11/17 [01:46<00:56,  9.45s/it]

...

OK    en_doc_25.pdf -> en_doc_25.md (323,683 chars)


Parsing PDFs:  94%|█████████▍| 16/17 [06:59<00:47, 47.31s/it]

OK    en_doc_26.pdf -> en_doc_26.md (74,023 chars)


Parsing PDFs: 100%|██████████| 17/17 [07:42<00:00, 27.18s/it]

OK    en_doc_27.pdf -> en_doc_27.md (152,126 chars)


In [9]:
# ============================================================
# SUMMARY
# ============================================================

print("\nDone.")
print(f"Processed: {processed}")
print(f"Skipped:   {skipped}")
print(f"Failed:    {failed}")

print(f"\nParser tier:    {PARSER_TIER}")
print(f"Parser version: {PARSER_VERSION}")


Done.
Processed: 8
Skipped:   9
Failed:    0

Parser tier:    agentic
Parser version: 2026-09-13
